# Tier 1 — Config & Logic Integration Tests (Free)

These tests exercise core logic using fakes — no API calls, no cost.
Each cell is independent after the setup cell.

**Run All** with `Kernel → Restart & Run All` for a full pass.

In [ ]:
# Setup — imports, path config, pricing bootstrap
import sys
from pathlib import Path

# Ensure notebooks/ is on path for test_helpers
nb_dir = str(Path(".").resolve())
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)

from test_helpers import (
    bootstrap_pricing, assert_pass, print_summary, run_async,
    UsageResult, PRICING, _lookup_pricing, estimate_cost,
    SessionAccumulator, build_api_call_metric,
    should_activate_tool_search,
    classify_turn,
    get_system_prompt,
    NoneCompactionStrategy, SummarizeCompactionStrategy, estimate_tokens,
    FakeStreamProvider, FakeTool, FakeProvider, BufferedChannel,
)

bootstrap_pricing()
print(f"Loaded pricing for {len(PRICING)} models")
print("Setup complete.")

## 1.1 Pricing Lookup
*Covers: [MANUAL-TEST-pricing.md](../documentation/docs/testing/MANUAL-TEST-pricing.md) Tests 2.1-2.3*

In [ ]:
# 1.1 — _lookup_pricing returns correct rates for all providers
print("1.1 Pricing Lookup")

# Anthropic
rates = _lookup_pricing("anthropic", "claude-sonnet-4-5-20250929")
assert_pass(rates is not None, "Anthropic sonnet-4-5 found")
assert_pass(rates[0] == 3.0, f"Anthropic sonnet-4-5 input price = {rates[0]}")
assert_pass(rates[1] == 15.0, f"Anthropic sonnet-4-5 output price = {rates[1]}")

# OpenAI
rates = _lookup_pricing("openai", "gpt-4o")
assert_pass(rates is not None, "OpenAI gpt-4o found")
assert_pass(rates[0] == 2.5, f"OpenAI gpt-4o input price = {rates[0]}")

# Gemini
rates = _lookup_pricing("gemini", "gemini-2.0-flash")
assert_pass(rates is not None, "Gemini 2.0-flash found")
assert_pass(rates[0] == 0.10, f"Gemini 2.0-flash input price = {rates[0]}")

# DeepSeek
rates = _lookup_pricing("deepseek", "deepseek-chat")
assert_pass(rates is not None, "DeepSeek chat found")
assert_pass(rates[0] == 0.27, f"DeepSeek chat input price = {rates[0]}")

# Unknown model
rates = _lookup_pricing("unknown", "nonexistent-model")
assert_pass(rates is None, "Unknown model returns None")

print("\n1.1 complete.")

## 1.2 SessionAccumulator — Basic Counters
*Covers: [MANUAL-TEST-pricing.md](../documentation/docs/testing/MANUAL-TEST-pricing.md) Tests 1.1-1.3*

In [ ]:
# 1.2 — add_api_call increments counters; reset clears all
print("1.2 SessionAccumulator — Basic Counters")

acc = SessionAccumulator(session_id="test-1.2")
usage = UsageResult(
    input_tokens=100, output_tokens=50,
    cache_read_input_tokens=20, cache_creation_input_tokens=10,
    duration_ms=500.0, provider="anthropic", model="claude-sonnet-4-5-20250929",
)

acc.add_api_call(usage, call_type="main", turn_number=1)
assert_pass(acc.total_api_calls == 1, "api_calls incremented to 1")
assert_pass(acc.total_input_tokens == 100, "input_tokens = 100")
assert_pass(acc.total_output_tokens == 50, "output_tokens = 50")
assert_pass(acc.total_cache_read_tokens == 20, "cache_read = 20")
assert_pass(acc.total_cache_creation_tokens == 10, "cache_create = 10")
assert_pass(acc.total_cost_usd > 0, f"cost > 0 (${acc.total_cost_usd:.6f})")
assert_pass(acc.total_duration_ms == 500.0, "duration_ms = 500")

# Add a second call
acc.add_api_call(usage, call_type="main", turn_number=2)
assert_pass(acc.total_api_calls == 2, "api_calls incremented to 2")
assert_pass(acc.total_input_tokens == 200, "input_tokens doubled")

# Reset
acc.reset("new-session")
assert_pass(acc.total_api_calls == 0, "reset: api_calls = 0")
assert_pass(acc.total_input_tokens == 0, "reset: input_tokens = 0")
assert_pass(acc.total_cost_usd == 0.0, "reset: cost = 0")
assert_pass(acc.session_id == "new-session", "reset: session_id updated")

print("\n1.2 complete.")

## 1.3 SessionAccumulator — Tool Call Tracking
*Covers: [MANUAL-TEST-pricing.md](../documentation/docs/testing/MANUAL-TEST-pricing.md) Tests 5.1-5.2*

In [ ]:
# 1.3 — Tool call tracking and error counting
print("1.3 SessionAccumulator — Tool Call Tracking")

acc = SessionAccumulator(session_id="test-1.3")

acc.add_tool_call("filesystem__read_file", is_error=False)
acc.add_tool_call("filesystem__read_file", is_error=False)
acc.add_tool_call("web__web_search", is_error=True)

assert_pass(acc.total_tool_calls == 3, "total_tool_calls = 3")
assert_pass(acc.total_tool_errors == 1, "total_tool_errors = 1")
assert_pass(acc.tool_call_counts["filesystem__read_file"] == 2, "read_file count = 2")
assert_pass(acc.tool_call_counts["web__web_search"] == 1, "web_search count = 1")

print("\n1.3 complete.")

## 1.4 SessionAccumulator — Model Subtotals
*Covers: [MANUAL-TEST-pricing.md](../documentation/docs/testing/MANUAL-TEST-pricing.md) Tests 4.1-4.2*

In [ ]:
# 1.4 — model_subtotals tracks multiple models separately
print("1.4 SessionAccumulator — Model Subtotals")

acc = SessionAccumulator(session_id="test-1.4")

usage_main = UsageResult(
    input_tokens=1000, output_tokens=500,
    provider="anthropic", model="claude-sonnet-4-5-20250929",
)
usage_cheap = UsageResult(
    input_tokens=200, output_tokens=100,
    provider="anthropic", model="claude-haiku-4-5-20251001",
)

acc.add_api_call(usage_main, call_type="main", turn_number=1)
acc.add_api_call(usage_cheap, call_type="routing", turn_number=2)
acc.add_api_call(usage_main, call_type="main", turn_number=3)

assert_pass(len(acc.model_subtotals) == 2, "2 model subtotals")

sonnet_key = "anthropic/claude-sonnet-4-5-20250929"
haiku_key = "anthropic/claude-haiku-4-5-20251001"

assert_pass(sonnet_key in acc.model_subtotals, "sonnet subtotal exists")
assert_pass(haiku_key in acc.model_subtotals, "haiku subtotal exists")
assert_pass(acc.model_subtotals[sonnet_key]["calls"] == 2, "sonnet: 2 calls")
assert_pass(acc.model_subtotals[haiku_key]["calls"] == 1, "haiku: 1 call")
assert_pass(
    acc.model_subtotals[sonnet_key]["cost_usd"] > acc.model_subtotals[haiku_key]["cost_usd"],
    "sonnet cost > haiku cost",
)

print("\n1.4 complete.")

## 1.5 Tool Search Activation Logic
*Covers: [MANUAL-TEST-tool-search-and-canonicalisation.md](../documentation/docs/testing/MANUAL-TEST-tool-search-and-canonicalisation.md) Tests 1-4, 7-12*

In [ ]:
# 1.5 — should_activate_tool_search: all provider×setting combos
print("1.5 Tool Search Activation Logic")

# Create a large tool set to exceed threshold for non-cache providers
big_tools = [
    {"name": f"tool_{i}", "description": f"A tool that does thing {i} " * 50,
     "input_schema": {"type": "object", "properties": {"x": {"type": "string"}}}}
    for i in range(60)
]
small_tools = [{"name": "t1", "description": "small", "input_schema": {"type": "object"}}]

# Explicit overrides
assert_pass(
    should_activate_tool_search("true", [], "any-model") is True,
    "setting='true' -> always active",
)
assert_pass(
    should_activate_tool_search("false", big_tools, "any-model") is False,
    "setting='false' -> always inactive",
)

# Cache-preserving providers (auto -> inactive regardless of tool count)
for prov in ["anthropic", "gemini", "deepseek"]:
    result = should_activate_tool_search("auto", big_tools, "claude-sonnet-4-5", provider=prov)
    assert_pass(result is False, f"auto + {prov} -> inactive (cache-preserving)")

# OpenAI auto — use auto:1 (1% threshold) so our test tool set exceeds it
result = should_activate_tool_search("auto:1", big_tools, "gpt-4o", provider="openai")
assert_pass(result is True, "auto:1 + openai + big tools -> active")

result = should_activate_tool_search("auto:1", small_tools, "gpt-4o", provider="openai")
assert_pass(result is False, "auto:1 + openai + small tools -> inactive")

# Unknown provider auto — threshold heuristic applies
result = should_activate_tool_search("auto:1", big_tools, "gpt-4o", provider="custom")
assert_pass(result is True, "auto:1 + unknown provider + big tools -> active")

print("\n1.5 complete.")

## 1.6 Per-Turn Routing — classify_turn
*Covers: [MANUAL-TEST-per-turn-routing.md](../documentation/docs/testing/MANUAL-TEST-per-turn-routing.md) Tests 1-5, 7*

In [ ]:
# 1.6 — classify_turn: all routing rules
print("1.6 Per-Turn Routing -- classify_turn")

kw = ["design", "architect", "analyze", "debug", "refactor"]

# Rule: tool_result_continuation (iteration > 0 -> cheap)
c = classify_turn(
    user_message="hello", has_tools=True, turn_iteration=1,
    turn_number=2, max_user_chars=200, short_followup_chars=50,
    complexity_keywords=kw,
)
assert_pass(c.use_cheap_model is True, "tool_result_continuation -> cheap")
assert_pass(c.rule == "tool_result_continuation", f"rule = {c.rule}")

# Rule: complexity_guard (keyword detected -> main model)
c = classify_turn(
    user_message="Please analyze this code", has_tools=True, turn_iteration=0,
    turn_number=2, max_user_chars=200, short_followup_chars=50,
    complexity_keywords=kw,
)
assert_pass(c.use_cheap_model is False, "complexity_guard -> main model")
assert_pass(c.rule == "complexity_guard", f"rule = {c.rule}")

# Rule: short_conversational (no tools, short message -> cheap)
c = classify_turn(
    user_message="thanks", has_tools=False, turn_iteration=0,
    turn_number=1, max_user_chars=200, short_followup_chars=50,
    complexity_keywords=kw,
)
assert_pass(c.use_cheap_model is True, "short_conversational -> cheap")
assert_pass(c.rule == "short_conversational", f"rule = {c.rule}")

# Rule: short_followup (turn > 1, short message -> cheap)
c = classify_turn(
    user_message="ok", has_tools=True, turn_iteration=0,
    turn_number=3, max_user_chars=200, short_followup_chars=50,
    complexity_keywords=kw,
)
assert_pass(c.use_cheap_model is True, "short_followup -> cheap")
assert_pass(c.rule == "short_followup", f"rule = {c.rule}")

# Rule: default (long message, turn 1 -> main model)
c = classify_turn(
    user_message="Please write a comprehensive report about the quarterly results" * 3,
    has_tools=True, turn_iteration=0,
    turn_number=1, max_user_chars=200, short_followup_chars=50,
    complexity_keywords=kw,
)
assert_pass(c.use_cheap_model is False, "default -> main model")
assert_pass(c.rule == "default", f"rule = {c.rule}")

# First turn with long message always uses main model
c = classify_turn(
    user_message="Please read this long document and summarize it for me in detail" * 2,
    has_tools=True, turn_iteration=0,
    turn_number=1, max_user_chars=200, short_followup_chars=50,
    complexity_keywords=kw,
)
assert_pass(c.use_cheap_model is False, "first turn + long message -> main model")

print("\n1.6 complete.")

## 1.7 Per-Turn Routing — Config Validation
*Covers: [MANUAL-TEST-per-turn-routing.md](../documentation/docs/testing/MANUAL-TEST-per-turn-routing.md) Test 6*

In [ ]:
# 1.7 — per_turn_routing_enabled=True without model should raise
print("1.7 Per-Turn Routing — Config Validation")

from micro_x_agent_loop.agent_config import AgentConfig

try:
    cfg = AgentConfig(per_turn_routing_enabled=True, per_turn_routing_model="")
    # The validation happens at agent startup, not config construction.
    # Check that the config allows constructing with empty model
    # (validation is in agent.py __init__)
    assert_pass(
        cfg.per_turn_routing_enabled is True and cfg.per_turn_routing_model == "",
        "Config allows routing_enabled=True with empty model (validated at agent startup)",
    )
except Exception as e:
    assert_pass(True, f"Config validation raised: {e}")

print("\n1.7 complete.")

## 1.8 Compaction Strategy — None
*Covers: [MANUAL-TEST-compaction-strategy.md](../documentation/docs/testing/MANUAL-TEST-compaction-strategy.md) Test 7*

In [ ]:
# 1.8 — NoneCompactionStrategy is a no-op
print("1.8 Compaction Strategy — None")

strategy = NoneCompactionStrategy()
messages = [
    {"role": "user", "content": "hello"},
    {"role": "assistant", "content": "hi"},
]
result = run_async(strategy.maybe_compact(messages))
assert_pass(result is messages, "NoneCompactionStrategy returns messages unchanged")
assert_pass(len(result) == 2, "message count preserved")

print("\n1.8 complete.")

## 1.9 Compaction Strategy — Summarize Trigger & Tool Pair Preservation
*Covers: [MANUAL-TEST-compaction-strategy.md](../documentation/docs/testing/MANUAL-TEST-compaction-strategy.md) Tests 1, 4*

In [ ]:
# 1.9 — SummarizeCompactionStrategy triggers above threshold, preserves tool pairs
print("1.9 Compaction Strategy -- Summarize Trigger & Tool Pair Preservation")

# Build a message list that exceeds a low threshold
long_text = "This is a test message with enough content to generate tokens. " * 100
messages = [
    {"role": "user", "content": "initial request"},
    {"role": "assistant", "content": long_text},
    {"role": "user", "content": long_text},
    {"role": "assistant", "content": long_text},
    {"role": "user", "content": long_text},
    {"role": "assistant", "content": [{"type": "text", "text": "Here's the answer."}]},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "t1", "content": "ok"}]},
    {"role": "assistant", "content": [{"type": "text", "text": "Done."}]},
]

est = estimate_tokens(messages)
print(f"  Estimated tokens: {est:,}")

# Use FakeProvider for the summarization call
fake_prov = FakeProvider(summary_text="Summary of conversation.")
compaction_called = []

def on_compaction(usage, before, after, count):
    compaction_called.append((before, after, count))

strategy = SummarizeCompactionStrategy(
    provider=fake_prov,
    model="test-model",
    threshold_tokens=100,  # Very low to force trigger
    protected_tail_messages=2,
    on_compaction_completed=on_compaction,
)

result = run_async(strategy.maybe_compact(messages))

assert_pass(len(result) < len(messages), f"messages compacted: {len(messages)} -> {len(result)}")
assert_pass(len(compaction_called) == 1, "compaction callback fired once")
assert_pass("[CONTEXT SUMMARY]" in result[0]["content"], "summary marker present")

# Check role alternation in result
for i in range(1, len(result)):
    assert_pass(
        result[i]["role"] != result[i-1]["role"],
        f"role alternation at position {i}: {result[i-1]['role']} -> {result[i]['role']}",
    )

print("\n1.9 complete.")

## 1.10 Compaction — Role Alternation After Compaction
*Covers: [MANUAL-TEST-compaction-strategy.md](../documentation/docs/testing/MANUAL-TEST-compaction-strategy.md) Test 8*

In [ ]:
# 1.10 — Role alternation is correct after compaction
print("1.10 Compaction -- Role Alternation")

long_text = "Content " * 200
messages = [
    {"role": "user", "content": "start"},
    {"role": "assistant", "content": long_text},
    {"role": "user", "content": long_text},
    {"role": "assistant", "content": long_text},
    {"role": "user", "content": "final question"},
    {"role": "assistant", "content": "final answer"},
]

fake_prov = FakeProvider(summary_text="Conversation summary.")
strategy = SummarizeCompactionStrategy(
    provider=fake_prov,
    model="test-model",
    threshold_tokens=50,
    protected_tail_messages=2,
)

result = run_async(strategy.maybe_compact(messages))

# Verify strict user/assistant alternation
assert_pass(result[0]["role"] == "user", "first message is user")
roles = [m["role"] for m in result]
for i in range(1, len(roles)):
    assert_pass(
        roles[i] != roles[i-1],
        f"alternation ok: pos {i-1}={roles[i-1]} -> pos {i}={roles[i]}",
    )

print("\n1.10 complete.")

## 1.11 Session Budget — No Budget Configured
*Covers: [MANUAL-TEST-session-budget.md](../documentation/docs/testing/MANUAL-TEST-session-budget.md) Test 4*

In [ ]:
# 1.11 — No budget configured = no warnings
print("1.11 Session Budget — No Budget Configured")

acc = SessionAccumulator(session_id="test-1.11")
usage = UsageResult(
    input_tokens=10000, output_tokens=5000,
    provider="anthropic", model="claude-sonnet-4-5-20250929",
)
acc.add_api_call(usage, call_type="main", turn_number=1)

# With budget=0, format_toolbar should not show budget percentage
toolbar = acc.format_toolbar(budget_usd=0.0)
assert_pass("/" not in toolbar.split("\u2502")[0], "no budget fraction in toolbar when budget=0")
assert_pass("%" not in toolbar.split("\u2502")[0], "no percentage in toolbar when budget=0")

print("\n1.11 complete.")

## 1.12 Concise Output Directive
*Covers: [MANUAL-TEST-concise-output.md](../documentation/docs/testing/MANUAL-TEST-concise-output.md) Test 1*

In [ ]:
# 1.12 — get_system_prompt with concise_output_enabled contains the directive
print("1.12 Concise Output Directive")

prompt_on = get_system_prompt(concise_output_enabled=True)
prompt_off = get_system_prompt(concise_output_enabled=False)

assert_pass("Minimize output tokens" in prompt_on, "concise directive present when enabled")
assert_pass("Minimize output tokens" not in prompt_off, "concise directive absent when disabled")
assert_pass("200 words" in prompt_on, "200-word target mentioned")

print("\n1.12 complete.")

## 1.13 Stage2 Model — Metrics Separation
*Covers: [MANUAL-TEST-stage2-model-haiku.md](../documentation/docs/testing/MANUAL-TEST-stage2-model-haiku.md) Test 3*

In [ ]:
# 1.13 — Metrics correctly separate Stage2 calls from main model
print("1.13 Stage2 Model — Metrics Separation")

acc = SessionAccumulator(session_id="test-1.13")

usage_main = UsageResult(
    input_tokens=1000, output_tokens=500,
    provider="anthropic", model="claude-sonnet-4-5-20250929",
)
usage_stage2 = UsageResult(
    input_tokens=300, output_tokens=50,
    provider="anthropic", model="claude-haiku-4-5-20251001",
)

acc.add_api_call(usage_main, call_type="main", turn_number=1)
acc.add_api_call(usage_stage2, call_type="stage2_classification", turn_number=1)
acc.add_api_call(usage_main, call_type="main", turn_number=2)

assert_pass(len(acc.model_subtotals) == 2, "2 distinct models tracked")

sonnet_sub = acc.model_subtotals["anthropic/claude-sonnet-4-5-20250929"]
haiku_sub = acc.model_subtotals["anthropic/claude-haiku-4-5-20251001"]

assert_pass(sonnet_sub["calls"] == 2, "sonnet: 2 main calls")
assert_pass(haiku_sub["calls"] == 1, "haiku: 1 stage2 call")
assert_pass(haiku_sub["input_tokens"] == 300, "haiku input tokens = 300")

# Check api_call_log has call_type info
stage2_logs = [c for c in acc.api_call_log if c["call_type"] == "stage2_classification"]
assert_pass(len(stage2_logs) == 1, "1 stage2 entry in api_call_log")
assert_pass(stage2_logs[0]["model"] == "claude-haiku-4-5-20251001", "stage2 log has haiku model")

print("\n1.13 complete.")

## 1.14 build_api_call_metric — Required Fields
*Covers: [MANUAL-TEST-cost-reconciliation.md](../documentation/docs/testing/MANUAL-TEST-cost-reconciliation.md) Test A.1*

In [ ]:
# 1.14 — build_api_call_metric returns all required fields
print("1.14 build_api_call_metric — Required Fields")

usage = UsageResult(
    input_tokens=500, output_tokens=200,
    cache_read_input_tokens=100, cache_creation_input_tokens=50,
    duration_ms=1200.0, time_to_first_token_ms=150.0,
    provider="anthropic", model="claude-sonnet-4-5-20250929",
    message_count=5, tool_schema_count=10, stop_reason="end_turn",
)

metric = build_api_call_metric(usage, session_id="s1", turn_number=1, call_type="main")

required_fields = [
    "type", "timestamp", "session_id", "turn_number", "call_type",
    "provider", "model", "input_tokens", "output_tokens",
    "cache_creation_input_tokens", "cache_read_input_tokens",
    "message_count", "tool_schema_count", "stop_reason",
    "duration_ms", "time_to_first_token_ms", "estimated_cost_usd",
]

for field in required_fields:
    assert_pass(field in metric, f"field '{field}' present")

assert_pass(metric["type"] == "api_call", "type = api_call")
assert_pass(metric["session_id"] == "s1", "session_id matches")
assert_pass(metric["estimated_cost_usd"] > 0, f"cost > 0 (${metric['estimated_cost_usd']:.6f})")
assert_pass(metric["input_tokens"] == 500, "input_tokens = 500")

# With routing info
metric_r = build_api_call_metric(
    usage, session_id="s1", turn_number=1, call_type="routing",
    routing_rule="short_followup", routing_reason="short message",
)
assert_pass("routing_rule" in metric_r, "routing_rule present when specified")
assert_pass(metric_r["routing_rule"] == "short_followup", "routing_rule value correct")

print("\n1.14 complete.")

## Summary

In [ ]:
print_summary()